# Robot Learning Selection Task

Runs in Colab (CPU only). Clone or upload the repo folder, then set `ROOT` below.

Read `README.md` for the task and `RESULTS.md` for what was tried and the results.

In [ ]:
!pip -q install numpy matplotlib
# !git clone <your fork of robotics_learning_group_selection>
import sys
ROOT = '/content/robotics_learning_group_selection'  # change if needed
sys.path.insert(0, ROOT)

## 1. Baseline (provided)

In [ ]:
from src.env import make_dataset
from src.policy import LinearPolicy
from src.evaluate import evaluate, make_scenarios

X, Y = make_dataset(n=4000, seed=42)
public = make_scenarios(300, seed=1)
baseline = LinearPolicy(X, Y)
result = evaluate(baseline, public)

for k in ('success_rate', 'mean_final_distance', 'mean_steps', 'collision_episode_rate'):
    print(f'{k}:', result[k])

In [ ]:
from src.plotting import plot_trajectory

plot_trajectory(result['trajectories'][0], title='Baseline rollout (example)');

## 2. StudentPolicy: symmetric BC prior + contact mapping + replanning

`act(obs)` still only sees `[dx, dy]`. Collisions are detected from the observation stream (a blocked step leaves `obs` unchanged), contact points are stored on a goal-centred grid, and a cost-to-go field picks the action whose real 0.08 end point is best.

In [ ]:
from src.policy import StudentPolicy

student = StudentPolicy(X, Y)
res_s = evaluate(student, public)
for k in ('success_rate', 'mean_final_distance', 'mean_steps', 'collision_episode_rate'):
    print(f'{k}: baseline {result[k]:.3f}   student {res_s[k]:.3f}')

## 3. Held-out-like stress test (more blocks, noise, shorter horizon)

In [ ]:
hard = make_scenarios(300, seed=21, n_blocks=(6, 8))
kw = dict(noise_std=0.01, action_noise_std=0.1, max_steps=40)
for name, pol in [('baseline', LinearPolicy(X, Y)), ('student', StudentPolicy(X, Y))]:
    r = evaluate(pol, hard, **kw)
    print(f"{name:9s} success {r['success_rate']:.3f}  steps {r['mean_steps']:.1f}  "
          f"collision-episode rate {r['collision_episode_rate']:.3f}")

## 4. Same scenario, both policies

In [ ]:
import numpy as np
fail = next(i for i, t in enumerate(result['trajectories'])
            if np.linalg.norm(t['positions'][-1] - t['target']) >= 0.05)
plot_trajectory(result['trajectories'][fail], title=f'Baseline, scenario {fail}');
plot_trajectory(res_s['trajectories'][fail], title=f'Student, scenario {fail}');

## 5. Reproduce all experiments and figures

`experiments/run_experiments.py` (~10 min on 2 CPUs) writes `results/results.json`; `experiments/make_figures.py` writes `results/figures/` and `results/summary.md`.

In [ ]:
# !cd $ROOT && python experiments/run_experiments.py && python experiments/make_figures.py